# RAG-Based Question Answering with Chunking and Vector Search



This notebook compares five ways to organize a small Problem–Solution dataset, searches for relevant text using embeddings and FAISS, and uses Groq to generate an answer from the retrieved context.

.

## 1. Install required packages

Run this cell once if the packages are not installed in your Python environment. If installation finishes, continue to the imports cell.

In [1]:
%pip install pandas numpy openpyxl sentence-transformers faiss-cpu groq

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 2. Import libraries and load the dataset

In [2]:
import os
import re
import numpy as np
import pandas as pd
import faiss
from getpass import getpass
from sentence_transformers import SentenceTransformer
from groq import Groq

DATA_FILE = "Data.xml.xlsx"

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Put it in the same folder as this notebook."
    )

df = pd.read_excel(DATA_FILE)
df.columns = df.columns.str.strip()

required_columns = {"Problem", "Solution"}
if not required_columns.issubset(df.columns):
    raise ValueError("The Excel file must contain 'Problem' and 'Solution' columns.")

df = df.dropna(subset=["Problem", "Solution"]).reset_index(drop=True)
df["Problem"] = df["Problem"].astype(str).str.strip()
df["Solution"] = df["Solution"].astype(str).str.strip()
df = df[(df["Problem"] != "") & (df["Solution"] != "")].reset_index(drop=True)

print(f"Dataset loaded: {len(df)} Problem–Solution records")
display(df.head())
print("DataFrame exists:", "df" in globals())
print("Columns:", df.columns.tolist())
print("Number of records:", len(df))

Dataset loaded: 86 Problem–Solution records


,Problem,Solution
0,Python list index error,An IndexError occurs when accessing a list pos...
1,Python KeyError in dictionary,A KeyError occurs when a requested dictionary ...
2,Python module not found,A ModuleNotFoundError usually means the packag...
3,Python virtual environment activation,Create a virtual environment with python -m ve...
4,Python file path error,A FileNotFoundError occurs when the path is in...


DataFrame exists: True
Columns: ['Problem', 'Solution']
Number of records: 86


## 3. Load the embedding model and set up shared helper functions

The model converts text into numerical vectors. FAISS searches those vectors. We load the model only once and reuse it for every method.

In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

def make_document(row):
    return f"Problem: {row['Problem']}\nSolution: {row['Solution']}"

def build_search_index(chunks):
    texts = [item["document"] for item in chunks]
    vectors = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index

def search_chunks(query, chunks, index, top_k=5):
    if not isinstance(query, str) or not query.strip():
        return []

    query_vector = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    top_k = min(top_k, index.ntotal)
    scores, positions = index.search(query_vector, top_k)

    results = []
    for rank, (score, position) in enumerate(zip(scores[0], positions[0]), start=1):
        if position < 0:
            continue
        item = chunks[int(position)]
        results.append({
            "rank": rank,
            "score": float(score),
            "document": item["document"],
            "metadata": item["metadata"]
        })
    return results

# Keep the original Problem–Solution pair as metadata for evaluation.
records = [
    {
        "problem": row["Problem"],
        "solution": row["Solution"],
        "document": make_document(row)
    }
    for _, row in df.iterrows()
]
print(f"Prepared {len(records)} source documents.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Prepared 86 source documents.


## 4. Semantic chunking

Semantic chunking groups nearby sentences when their embeddings are similar. Each generated chunk keeps the source problem title in its metadata.

In [4]:

import numpy as np
import faiss
import re
from sentence_transformers import SentenceTransformer

# 1. Load the semantic embedding model
semantic_embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


# 2. Split text into sentences
def semantic_split_sentences(text):
    text = str(text).strip()

    if not text:
        return []

    sentences = re.split(r'(?<=[.!?])\s+', text)

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


# 3. Create semantic chunks
def create_semantic_chunks(records, threshold=0.55, max_sentences=4):
    semantic_chunks = []

    for record in records:
        problem = str(record["problem"]).strip()
        solution = str(record["solution"]).strip()

        text = f"Problem: {problem}\nSolution: {solution}"
        sentences = semantic_split_sentences(text)

        if not sentences:
            continue

        # Keep short records as one chunk
        if len(sentences) <= max_sentences:
            semantic_chunks.append({
                "document": text,
                "metadata": {
                    "problem": problem,
                    "solution": solution
                }
            })
            continue

        # Embed sentences for semantic grouping
        sentence_embeddings = semantic_embedding_model.encode(
            sentences,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        ).astype(np.float32)

        current_group = [sentences[0]]

        for i in range(1, len(sentences)):
            similarity = float(
                np.dot(
                    sentence_embeddings[i - 1],
                    sentence_embeddings[i]
                )
            )

            if (
                similarity < threshold
                or len(current_group) >= max_sentences
            ):
                semantic_chunks.append({
                    "document": " ".join(current_group),
                    "metadata": {
                        "problem": problem,
                        "solution": solution
                    }
                })

                current_group = [sentences[i]]
            else:
                current_group.append(sentences[i])

        # Save the final group
        if current_group:
            semantic_chunks.append({
                "document": " ".join(current_group),
                "metadata": {
                    "problem": problem,
                    "solution": solution
                }
            })

    return semantic_chunks


# 4. Create semantic chunks
print("Starting Semantic Chunking...")

semantic_chunks = create_semantic_chunks(
    records,
    threshold=0.55,
    max_sentences=4
)

print("Semantic chunks created:", len(semantic_chunks))


# 5. Generate embeddings for the chunks
semantic_documents = [
    chunk["document"] for chunk in semantic_chunks
]

semantic_embeddings = semantic_embedding_model.encode(
    semantic_documents,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype(np.float32)


# 6. Build the FAISS index
semantic_index = faiss.IndexFlatIP(
    semantic_embeddings.shape[1]
)

semantic_index.add(semantic_embeddings)

print("Semantic embedding shape:", semantic_embeddings.shape)
print("Semantic FAISS vectors:", semantic_index.ntotal)


# 7. Semantic search with the shared result format
def semantic_search(query, top_k=5):
    if not isinstance(query, str) or not query.strip():
        return []

    query_embedding = semantic_embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    k = min(top_k, semantic_index.ntotal)

    if k <= 0:
        return []

    scores, indices = semantic_index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        results.append({
            "rank": len(results) + 1,
            "score": float(score),
            "document": semantic_documents[int(idx)],
            "metadata": semantic_chunks[int(idx)]["metadata"]
        })

    return results


# 8. Display semantic search results
test_query = "How do I fix a Python list index error?"

semantic_results = semantic_search(test_query, top_k=5)

print("\nSemantic Search Results:")

for result in semantic_results:
    print(f"\nRank: {result['rank']}")
    print(f"Similarity: {result['score']:.4f}")
    print(f"Problem: {result['metadata']['problem']}")
    print(result["document"])
    print("-" * 50)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Starting Semantic Chunking...
Semantic chunks created: 86


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Semantic embedding shape: (86, 384)
Semantic FAISS vectors: 86

Semantic Search Results:

Rank: 1
Similarity: 0.8265
Problem: Python list index error
Problem: Python list index error
Solution: An IndexError occurs when accessing a list position that does not exist. Check the list length and ensure the index is within the valid range.
--------------------------------------------------

Rank: 2
Similarity: 0.5269
Problem: Python KeyError in dictionary
Problem: Python KeyError in dictionary
Solution: A KeyError occurs when a requested dictionary key does not exist. Use get(), check whether the key exists, or provide a default value.
--------------------------------------------------

Rank: 3
Similarity: 0.4315
Problem: Python file path error
Problem: Python file path error
Solution: A FileNotFoundError occurs when the path is incorrect or the file does not exist. Verify the working directory and use the correct relative or absolute path.
--------------------------------------------------


## 5. Fixed chunking

Fixed chunking splits text into chunks of a set character length. A chunk may split a sentence or a Problem–Solution pair, so source metadata is retained.

In [5]:
def split_fixed(text, chunk_size=100, overlap=0):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Use chunk_size > 0 and 0 <= overlap < chunk_size.")

    chunks = []
    start = 0
    while start < len(text):
        piece = text[start:start + chunk_size].strip()
        if piece:
            chunks.append(piece)
        start += chunk_size - overlap
    return chunks

fixed_chunks = []
for record in records:
    for piece in split_fixed(record["document"], chunk_size=100, overlap=0):
        fixed_chunks.append({
            "document": piece,
            "metadata": {
                "problem": record["problem"],
                "solution": record["solution"]
            }
        })

fixed_index = build_search_index(fixed_chunks)
print(f"Fixed chunks: {len(fixed_chunks)}")

Fixed chunks: 177


## 6. Recursive chunking

This beginner-friendly recursive splitter tries paragraph breaks first, then sentence breaks, and finally character boundaries when a piece is still too long.

In [6]:

def recursive_split(text, chunk_size=200, overlap=20):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError(
            "Use chunk_size > 0 and 0 <= overlap < chunk_size."
        )

    text = str(text).strip()

    if not text:
        return []

    if len(text) <= chunk_size:
        return [text]

    # Prefer paragraph, line, sentence, then word boundaries.
    separators = ["\n\n", "\n", ". ", " "]

    for separator in separators:
        parts = text.split(separator)

        if len(parts) <= 1:
            continue

        pieces = []
        current = ""

        for part in parts:
            candidate = (
                part if not current
                else current + separator + part
            )

            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    pieces.append(current)

                # Split unusually long segments safely.
                if len(part) > chunk_size:
                    start = 0
                    step = chunk_size - overlap

                    while start < len(part):
                        piece = part[start:start + chunk_size]
                        if piece:
                            pieces.append(piece)
                        start += step

                    current = ""
                else:
                    current = part

        if current:
            pieces.append(current)

        # Use this split only if every chunk respects the limit.
        if pieces and all(
            0 < len(piece) <= chunk_size for piece in pieces
        ):
            # Apply overlap between consecutive chunks.
            if overlap == 0:
                return pieces

            overlapped_pieces = [pieces[0]]

            for piece in pieces[1:]:
                previous = overlapped_pieces[-1]
                prefix = previous[-overlap:]
                overlapped_pieces.append(prefix + piece)

            return [
                piece[:chunk_size]
                for piece in overlapped_pieces
            ]

    # Final character-based fallback.
    step = chunk_size - overlap

    return [
        text[start:start + chunk_size]
        for start in range(0, len(text), step)
    ]


# Create recursive chunks
recursive_chunks = []

for record in records:
    pieces = recursive_split(
        record["document"],
        chunk_size=200,
        overlap=20
    )

    for piece in pieces:
        recursive_chunks.append({
            "document": piece,
            "metadata": {
                "problem": record["problem"],
                "solution": record["solution"]
            }
        })


# Build the existing Recursive FAISS index
recursive_index = build_search_index(recursive_chunks)

print(f"Recursive chunks: {len(recursive_chunks)}")
print(f"Recursive FAISS vectors: {recursive_index.ntotal}")


Recursive chunks: 91
Recursive FAISS vectors: 91


## 7. Documentation chunking

For this dataset, the simplest documentation strategy is to keep each full Problem–Solution pair together. This preserves the complete information for retrieval.

In [7]:
documentation_chunks = [
    {
        "document": record["document"],
        "metadata": {
            "problem": record["problem"],
            "solution": record["solution"]
        }
    }
    for record in records
]

documentation_index = build_search_index(documentation_chunks)
print(f"Documentation chunks: {len(documentation_chunks)}")

Documentation chunks: 86


## 8. Agentic chunking baseline

This simple baseline treats each complete Problem–Solution pair as a unit so that the system does not separate a problem from its solution. It is included as a comparison baseline; it is not a complex autonomous agent.

In [8]:

import re
import unicodedata
import numpy as np
import faiss

# Normalize problem titles to identify duplicate records
def normalize_title(title):
    title = unicodedata.normalize("NFKC", str(title))
    title = title.strip().lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    return " ".join(title.split())


# Prepare Agentic Retrieval documents from the original records
agentic_chunks = []

for record in records:
    agentic_chunks.append({
        "document": record["document"],
        "metadata": {
            "problem": record["problem"],
            "solution": record["solution"]
        }
    })

# Build the FAISS index for Agentic Retrieval
agentic_index = build_search_index(agentic_chunks)

# Agentic Retrieval configuration
AGENTIC_SIMILARITY_THRESHOLD = 0.60
AGENTIC_MAX_ATTEMPTS = 2


def agentic_retrieve(
    question,
    top_k=5,
    max_attempts=AGENTIC_MAX_ATTEMPTS
):
    """
    Rule-based iterative retrieval controller.

    Retrieves relevant records, checks the similarity threshold,
    refines the query if necessary, and rejects weak results.
    """

    if not isinstance(question, str) or not question.strip():
        return {
            "results": [],
            "attempts": [],
            "accepted": False,
            "reason": "Empty or invalid question"
        }

    attempts = []
    seen_problems = set()
    unique_results = []

    current_query = question.strip()
    accepted = False
    reason = "No results met the similarity threshold"

    for attempt_number in range(1, max_attempts + 1):

        retrieved = search_chunks(
            current_query,
            agentic_chunks,
            agentic_index,
            top_k=top_k
        )

        attempts.append({
            "attempt": attempt_number,
            "query": current_query,
            "results": retrieved
        })

        # Remove duplicate problem records across attempts
        for result in retrieved:
            problem_key = normalize_title(
                result["metadata"]["problem"]
            )

            if problem_key not in seen_problems:
                seen_problems.add(problem_key)
                unique_results.append(result)

        # Accept results only when the current query's best
        # similarity reaches the configured threshold
        if retrieved and retrieved[0]["score"] >= AGENTIC_SIMILARITY_THRESHOLD:
            accepted = True
            reason = (
                "Relevant results found above the similarity threshold"
            )
            break

        # Refine the query for the next attempt
        if attempt_number < max_attempts:
            current_query = (
                question.strip()
                + " error cause troubleshooting fix solution"
            )

    # Return results only when retrieval was accepted
    final_results = unique_results if accepted else []

    return {
        "results": final_results[:top_k],
        "attempts": attempts,
        "accepted": accepted,
        "reason": reason
    }


print("Agentic retrieval records:", len(agentic_chunks))
print("Similarity threshold:", AGENTIC_SIMILARITY_THRESHOLD)
print("Maximum attempts:", AGENTIC_MAX_ATTEMPTS)


Agentic retrieval records: 86
Similarity threshold: 0.6
Maximum attempts: 2


## 9. Compare retrieval methods

Ask one question and compare the top results from all five methods. This cell is for demonstration, not a full benchmark.

In [9]:

import pandas as pd

# Register the five retrieval methods and FAISS indexes
search_methods = {
    "Fixed": (fixed_chunks, fixed_index),
    "Semantic": (semantic_chunks, semantic_index),
    "Recursive": (recursive_chunks, recursive_index),
    "Documentation": (documentation_chunks, documentation_index),
    "Agentic": (agentic_chunks, agentic_index)
}


# Compare retrieval across all five methods
def compare_retrieval(question, top_k=3):

    for method_name, (chunks, index) in search_methods.items():

        print("\n" + "=" * 55)
        print(method_name)
        print("=" * 55)
        print("Question:", question)

        if method_name == "Agentic":

            output = agentic_retrieve(
                question,
                top_k=top_k
            )

            results = output["results"]

            print("Attempts made:", len(output["attempts"]))
            print("Accepted:", output["accepted"])
            print("Decision:", output["reason"])

            for attempt in output["attempts"]:
                print(
                    f"Attempt {attempt['attempt']}: "
                    f"{attempt['query']}"
                )

                if attempt["results"]:
                    print(
                        "Top similarity:",
                        f"{attempt['results'][0]['score']:.4f}"
                    )

        else:
            results = search_chunks(
                question,
                chunks,
                index,
                top_k=top_k
            )

        if not results:
            print("No relevant results accepted.")
            continue

        for rank, result in enumerate(results, start=1):

            print(
                f"\nRank {rank} | "
                f"Similarity: {result['score']:.4f}"
            )
            print("Problem:", result["metadata"]["problem"])
            print(result["document"][:300])
            print()


# Validate Agentic relevance-threshold decisions
threshold_test_questions = [
    {
        "question": "How do I fix a Python list index error?",
        "expected": True
    },
    {
        "question": "How do I resolve a Python module not found error?",
        "expected": True
    },
    {
        "question": "How do I fix a FileNotFoundError in Python?",
        "expected": True
    },
    {
        "question": "How do I handle missing values in Pandas?",
        "expected": True
    },
    {
        "question": "How do I activate a Python virtual environment?",
        "expected": True
    },
    {
        "question": "How can I bake a chocolate cake?",
        "expected": False
    },
    {
        "question": "What are the best tourist attractions in Paris?",
        "expected": False
    },
    {
        "question": "Explain the rules of cricket.",
        "expected": False
    },
    {
        "question": "How do I grow roses in my garden?",
        "expected": False
    },
    {
        "question": "What is the capital of Japan?",
        "expected": False
    }
]

threshold_results = []

for item in threshold_test_questions:

    output = agentic_retrieve(
        item["question"],
        top_k=3
    )

    actual = output["accepted"]
    expected = item["expected"]

    scores = [
        result["score"]
        for attempt in output["attempts"]
        for result in attempt["results"]
    ]

    threshold_results.append({
        "Question": item["question"],
        "Expected In Scope": expected,
        "Accepted": actual,
        "Correct Decision": actual == expected,
        "Attempts": len(output["attempts"]),
        "Best Similarity": max(scores) if scores else None
    })


threshold_df = pd.DataFrame(threshold_results)

print("\n" + "=" * 65)
print("AGENTIC RELEVANCE-THRESHOLD EVALUATION")
print("=" * 65)

display(threshold_df)

correct_decisions = int(
    threshold_df["Correct Decision"].sum()
)
total_questions = len(threshold_df)

print("\nThreshold validation summary")
print("Questions tested:", total_questions)
print("Correct decisions:", correct_decisions)

print(
    "Decision accuracy:",
    f"{correct_decisions / total_questions * 100:.1f}%"
)

print("\nFalse rejections:")
display(
    threshold_df[
        threshold_df["Expected In Scope"]
        & (~threshold_df["Accepted"])
    ]
)

print("\nFalse acceptances:")
display(
    threshold_df[
        (~threshold_df["Expected In Scope"])
        & threshold_df["Accepted"]
    ]
)



AGENTIC RELEVANCE-THRESHOLD EVALUATION


,Question,Expected In Scope,Accepted,Correct Decision,Attempts,Best Similarity
0,How do I fix a Python list index error?,True,True,True,1,0.826514
1,How do I resolve a Python module not found error?,True,True,True,1,0.762149
2,How do I fix a FileNotFoundError in Python?,True,True,True,1,0.827136
3,How do I handle missing values in Pandas?,True,True,True,1,0.682748
4,How do I activate a Python virtual environment?,True,True,True,1,0.608890
5,How can I bake a chocolate cake?,False,False,True,2,0.327497
6,What are the best tourist attractions in Paris?,False,False,True,2,0.290695
7,Explain the rules of cricket.,False,False,True,2,0.289478
8,How do I grow roses in my garden?,False,False,True,2,0.280984
9,What is the capital of Japan?,False,False,True,2,0.304301



Threshold validation summary
Questions tested: 10
Correct decisions: 10
Decision accuracy: 100.0%

False rejections:


,Question,Expected In Scope,Accepted,Correct Decision,Attempts,Best Similarity



False acceptances:


,Question,Expected In Scope,Accepted,Correct Decision,Attempts,Best Similarity


## 10. Generate an answer with Groq

This is the RAG part: retrieve relevant text, then ask the language model to answer using that context. Your API key is requested privately in the notebook and is not written into the file.

In [10]:

import getpass
from groq import Groq

# Step 1: Check existing RAG dependencies
if "search_methods" not in globals():
    raise NameError(
        "search_methods is missing. Run the existing chunking "
        "and FAISS setup cells first."
    )

if "search_chunks" not in globals():
    raise NameError(
        "search_chunks is missing. Run the existing retrieval "
        "function cell first."
    )

required_methods = [
    "Fixed",
    "Semantic",
    "Recursive",
    "Documentation",
    "Agentic"
]

missing_methods = [
    name for name in required_methods
    if name not in search_methods
]

if missing_methods:
    raise NameError(
        "Missing methods in search_methods: "
        + ", ".join(missing_methods)
        + ". Run their existing setup cells first."
    )


# Step 2: Ask for the Groq API key
api_key = getpass.getpass("Enter your Groq API key: ").strip()

if not api_key:
    raise ValueError("Groq API key is required.")

# Step 3: Initialize and verify Groq
groq_client = Groq(api_key=api_key)
GROQ_MODEL = "openai/gpt-oss-20b"

groq_client.models.list()
print("Groq API connected successfully.")


# Step 4: RAG question-answering function
SIMILARITY_THRESHOLD = 0.60

def ask_rag(question, method="Documentation", top_k=5):
    if not isinstance(question, str) or not question.strip():
        return "Please enter a valid question."

    if method not in search_methods:
        raise ValueError(f"Unknown chunking method: {method}")

    chunks, index = search_methods[method]

    results = search_chunks(
        question,
        chunks,
        index,
        top_k=top_k
    )

    relevant_results = [
        result for result in results
        if result.get("score") is not None
        and float(result["score"]) >= SIMILARITY_THRESHOLD
    ]

    if not relevant_results:
        return (
            "I couldn't find enough relevant information in "
            "the dataset to answer that question."
        )

    context = "\n\n".join(
        result["document"] for result in relevant_results
    )

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a technical support assistant. Answer the "
                    "current question using only the retrieved context. "
                    "Do not invent facts. If the context is insufficient, "
                    "reply: 'I couldn't find enough relevant information "
                    "in the dataset to answer that question.'"
                )
            },
            {
                "role": "user",
                "content": (
                    f"Retrieved context:\n{context}\n\n"
                    f"Question: {question}"
                )
            }
        ],
        temperature=0.2,
        max_tokens=800
    )

    answer = response.choices[0].message.content

    if not answer or not answer.strip():
        return "I couldn't generate an answer. Please try again."

    return answer.strip()


# Step 5: Interactive chatbot
def run_chatbot():
    print("\nRAG Chatbot is ready. Type 'exit' to quit.")

    while True:
        try:
            question = input("\nYou: ").strip()

            if question.lower() == "exit":
                print("\nChatbot closed by user.")
                break

            if not question:
                continue

            print("\nAssistant: Thinking...")

            answer = ask_rag(
                question=question,
                method="Documentation",
                top_k=5
            )

            print(f"\nAssistant: {answer}")

        except KeyboardInterrupt:
            print("\nChatbot interrupted and closed.")
            break

        except Exception as e:
            print(f"\nError: {e}")


# Step 6: Start the chatbot
run_chatbot()


Groq API connected successfully.

RAG Chatbot is ready. Type 'exit' to quit.

Assistant: Thinking...

Assistant: To fix a Python list index error, make sure the index you’re using is within the valid range of the list. Check the list’s length (e.g., `len(my_list)`) and ensure that any index you access is between `0` and `len(my_list) - 1`. If you need to access the last element, use `-1` or `len(my_list)-1`. Adjust your code so that it never tries to read or write beyond these bounds.

Assistant: Thinking...

Assistant: A KeyError happens when you try to access a dictionary key that isn’t present. To resolve it, you can:

1. **Use `dict.get()`** – this returns `None` (or a value you specify) instead of raising an error.  
   ```python
   value = my_dict.get('missing_key')          # returns None
   value = my_dict.get('missing_key', 'default')  # returns 'default'
   ```

2. **Check if the key exists first** – with an `if` statement or `in`.  
   ```python
   if 'missing_key' in my_dic

## 11. Simple retrieval evaluation

This small evaluation checks whether the expected problem title appears in the top five results. It measures title retrieval only; it does not prove that a generated answer is factually correct. Use a few questions whose expected titles you have checked against your dataset.

In [11]:

# Cell 11: Evaluate all retrieval methods using actual dataset titles

import pandas as pd

# Select 10 existing records from the original dataset.
# The questions and expected titles come directly from the dataset.
evaluation_records = df.drop_duplicates(
    subset=["Problem"]
).head(10)

test_questions = [
    str(problem).strip()
    for problem in evaluation_records["Problem"]
]

expected_titles = {
    question: question
    for question in test_questions
}

evaluation_rows = []

for question in test_questions:
    expected_normalized = normalize_title(
        expected_titles[question]
    )

    for method_name, (chunks, index) in search_methods.items():

        if method_name == "Agentic":
            output = agentic_retrieve(question, top_k=5)
            results = output["results"]
            accepted = output["accepted"]
            attempts = len(output["attempts"])

        else:
            results = search_chunks(
                question,
                chunks,
                index,
                top_k=index.ntotal
            )
            accepted = True
            attempts = 1

            # Keep only unique source problems.
            unique_results = []
            seen_problems = set()

            for result in results:
                problem_key = normalize_title(
                    result["metadata"]["problem"]
                )

                if problem_key not in seen_problems:
                    seen_problems.add(problem_key)
                    unique_results.append(result)

                if len(unique_results) >= 5:
                    break

            results = unique_results

        matching_rank = None
        matching_score = None

        if accepted:
            for rank, result in enumerate(results[:5], start=1):
                if (
                    normalize_title(result["metadata"]["problem"])
                    == expected_normalized
                ):
                    matching_rank = rank
                    matching_score = result["score"]
                    break

        recall_at_5 = int(matching_rank is not None)
        reciprocal_rank = (
            1 / matching_rank
            if matching_rank is not None
            else 0.0
        )

        evaluation_rows.append({
            "Question": question,
            "Method": method_name,
            "Expected Problem": expected_titles[question],
            "Accepted": accepted,
            "Attempts": attempts,
            "Matching Rank": matching_rank,
            "Similarity Score": matching_score,
            "Recall@5": recall_at_5,
            "Reciprocal Rank": reciprocal_rank
        })

evaluation_df = pd.DataFrame(evaluation_rows)

summary_df = (
    evaluation_df.groupby("Method", sort=False)
    .agg(
        Test_Questions=("Question", "count"),
        Successful_Retrievals=("Recall@5", "sum"),
        Recall_at_5=("Recall@5", "mean"),
        MRR=("Reciprocal Rank", "mean")
    )
    .reset_index()
)

summary_df["Recall_at_5"] = (
    summary_df["Recall_at_5"] * 100
).round(2)

summary_df["MRR"] = summary_df["MRR"].round(4)

print("Evaluation questions:", len(test_questions))
print("Record-level retrieval evaluation")
display(evaluation_df)

print("\nSummary across retrieval methods")
display(summary_df)


Evaluation questions: 10
Record-level retrieval evaluation


,Question,Method,Expected Problem,Accepted,Attempts,Matching Rank,Similarity Score,Recall@5,Reciprocal Rank
0,Python list index error,Fixed,Python list index error,True,1,1,0.868633,1,1.0
1,Python list index error,Semantic,Python list index error,True,1,1,0.811266,1,1.0
2,Python list index error,Recursive,Python list index error,True,1,1,0.811266,1,1.0
3,Python list index error,Documentation,Python list index error,True,1,1,0.811266,1,1.0
4,Python list index error,Agentic,Python list index error,True,1,1,0.811266,1,1.0
5,Python KeyError in dictionary,Fixed,Python KeyError in dictionary,True,1,1,0.902080,1,1.0
6,Python KeyError in dictionary,Semantic,Python KeyError in dictionary,True,1,1,0.862669,1,1.0
7,Python KeyError in dictionary,Recursive,Python KeyError in dictionary,True,1,1,0.862669,1,1.0
8,Python KeyError in dictionary,Documentation,Python KeyError in dictionary,True,1,1,0.862669,1,1.0
9,Python KeyError in dictionary,Agentic,Python KeyError in dictionary,True,1,1,0.862669,1,1.0



Summary across retrieval methods


,Method,Test_Questions,Successful_Retrievals,Recall_at_5,MRR
0,Fixed,10,10,100.0,1.0
1,Semantic,10,10,100.0,1.0
2,Recursive,10,10,100.0,1.0
3,Documentation,10,10,100.0,1.0
4,Agentic,10,10,100.0,1.0


In [12]:

# Step 1: Inspect the retrieved context for a question

test_question = "How do I fix a Python list index error?"

method = "Documentation"
top_k = 5

chunks, index = search_methods[method]

results = search_chunks(
    test_question,
    chunks,
    index,
    top_k=top_k
)

print("QUESTION:")
print(test_question)

print(f"\nCHUNKING METHOD: {method}")
print(f"RETRIEVED CHUNKS: {len(results)}")

for i, result in enumerate(results, start=1):
    print(f"\n{'=' * 60}")
    print(f"CHUNK {i}")

    if "score" in result:
        print("Similarity score:", result["score"])

    print(result["document"])


QUESTION:
How do I fix a Python list index error?

CHUNKING METHOD: Documentation
RETRIEVED CHUNKS: 5

CHUNK 1
Similarity score: 0.826514482498169
Problem: Python list index error
Solution: An IndexError occurs when accessing a list position that does not exist. Check the list length and ensure the index is within the valid range.

CHUNK 2
Similarity score: 0.5268566012382507
Problem: Python KeyError in dictionary
Solution: A KeyError occurs when a requested dictionary key does not exist. Use get(), check whether the key exists, or provide a default value.

CHUNK 3
Similarity score: 0.4314883351325989
Problem: Python file path error
Solution: A FileNotFoundError occurs when the path is incorrect or the file does not exist. Verify the working directory and use the correct relative or absolute path.

CHUNK 4
Similarity score: 0.39793068170547485
Problem: Pandas missing column
Solution: A KeyError can occur when a DataFrame column name is incorrect. Print df.columns, remove extra whitespa

In [13]:

# Answer Quality Evaluation: Correctness and Faithfulness

import json
import re
import pandas as pd

# Select five real Problem-Solution records from the dataset
answer_test_records = (
    df.drop_duplicates(subset=["Problem"])
    .head(5)
    .reset_index(drop=True)
)

answer_test_questions = [
    str(problem).strip()
    for problem in answer_test_records["Problem"]
]


# Retrieve the reference solution using the actual dataset title
def get_reference_solution(question):
    matched_rows = df[
        df["Problem"].apply(normalize_title)
        == normalize_title(question)
    ]

    if matched_rows.empty:
        raise ValueError(
            f"Reference problem not found in dataset: {question}"
        )

    return str(matched_rows.iloc[0]["Solution"])


# Evaluate correctness and faithfulness using the Groq model
def evaluate_generated_answer(question, answer, reference, context):
    prompt = f"""
You are evaluating a Retrieval-Augmented Generation (RAG) answer.

QUESTION:
{question}

REFERENCE SOLUTION FROM THE DATASET:
{reference}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}

Evaluate two dimensions on a scale from 1 to 5.

1. Correctness:
Does the generated answer correctly address the question and agree
with the reference solution? Equivalent correct explanations are allowed.

2. Faithfulness:
Are the factual claims supported by the retrieved context?
Penalize unsupported claims and contradictions.

Scoring:
1 = very poor
2 = poor
3 = partially satisfactory
4 = good
5 = excellent

Return only a valid JSON object in this format:
{{
  "correctness": 1,
  "faithfulness": 1,
  "reason": "Brief explanation"
}}
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a careful evaluator of RAG answer quality. "
                    "Use only the supplied question, reference, context, "
                    "and answer. Return valid JSON."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    raw_result = response.choices[0].message.content.strip()

    # Remove optional Markdown code fences
    raw_result = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw_result
    ).strip()

    result = json.loads(raw_result)

    correctness = int(result["correctness"])
    faithfulness = int(result["faithfulness"])

    if not (
        1 <= correctness <= 5
        and 1 <= faithfulness <= 5
    ):
        raise ValueError(
            "Evaluator returned a score outside 1–5."
        )

    return {
        "Correctness": correctness,
        "Faithfulness": faithfulness,
        "Evaluation Reason": result.get("reason", "")
    }


# Run answer-quality evaluation
answer_quality_rows = []

for question in answer_test_questions:
    print("Evaluating:", question)

    reference = get_reference_solution(question)

    # Generate an answer using the existing RAG chatbot
    answer = ask_rag(
        question,
        method="Documentation",
        top_k=5
    )

    if not isinstance(answer, str) or not answer.strip():
        answer_quality_rows.append({
            "Question": question,
            "Generated Answer": str(answer),
            "Correctness": None,
            "Faithfulness": None,
            "Evaluation Reason": "No usable answer was generated."
        })
        continue

    # Retrieve context from the Documentation method
    chunks, index = search_methods["Documentation"]

    retrieved_results = search_chunks(
        question,
        chunks,
        index,
        top_k=5
    )

    # Apply the chatbot's similarity threshold
    relevant_results = [
        result
        for result in retrieved_results
        if result["score"] >= SIMILARITY_THRESHOLD
    ]

    context = "\n\n".join(
        result["document"]
        for result in relevant_results
    )

    if not context:
        answer_quality_rows.append({
            "Question": question,
            "Generated Answer": answer,
            "Correctness": None,
            "Faithfulness": None,
            "Evaluation Reason": (
                "No context passed the similarity threshold; "
                "answer quality was not scored."
            )
        })
        continue

    try:
        scores = evaluate_generated_answer(
            question=question,
            answer=answer,
            reference=reference,
            context=context
        )

        answer_quality_rows.append({
            "Question": question,
            "Generated Answer": answer,
            **scores
        })

    except Exception as error:
        answer_quality_rows.append({
            "Question": question,
            "Generated Answer": answer,
            "Correctness": None,
            "Faithfulness": None,
            "Evaluation Reason": f"Evaluation failed: {error}"
        })


# Display results
answer_quality_df = pd.DataFrame(answer_quality_rows)

print("\nANSWER QUALITY EVALUATION")
display(answer_quality_df)

valid_scores = answer_quality_df.dropna(
    subset=["Correctness", "Faithfulness"]
)

if not valid_scores.empty:
    print("\nAverage scores out of 5")
    print(
        "Correctness:",
        round(valid_scores["Correctness"].mean(), 2)
    )
    print(
        "Faithfulness:",
        round(valid_scores["Faithfulness"].mean(), 2)
    )
    print("Answers evaluated:", len(valid_scores))
else:
    print("No answers received valid evaluation scores.")


Evaluating: Python list index error
Evaluating: Python KeyError in dictionary
Evaluating: Python module not found
Evaluating: Python virtual environment activation
Evaluating: Python file path error

ANSWER QUALITY EVALUATION


,Question,Generated Answer,Correctness,Faithfulness,Evaluation Reason
0,Python list index error,An **IndexError** occurs when you try to acces...,5,5,The generated answer accurately explains that ...
1,Python KeyError in dictionary,A **KeyError** in Python happens when you try ...,5,5,The answer accurately explains that a KeyError...
2,Python module not found,A ModuleNotFoundError usually means the packag...,5,5,The generated answer exactly matches the refer...
3,Python virtual environment activation,To activate a Python virtual environment in Po...,5,5,The answer accurately follows the reference so...
4,Python file path error,A **FileNotFoundError** in Python usually mean...,5,5,The generated answer accurately reflects the r...



Average scores out of 5
Correctness: 5.0
Faithfulness: 5.0
Answers evaluated: 5


## 12. Save the results

Run this cell after the evaluation cell. The CSV files will be saved in the notebook's current working directory.

In [14]:

# Cell 12: Save Evaluation Results

evaluation_df.to_csv("retrieval_evaluation_details.csv", index=False)
summary_df.to_csv("retrieval_evaluation_summary.csv", index=False)
answer_quality_df.to_csv("answer_quality_evaluation.csv", index=False)

print("Saved evaluation files:")
print("1. retrieval_evaluation_details.csv")
print("2. retrieval_evaluation_summary.csv")
print("3. answer_quality_evaluation.csv")


Saved evaluation files:
1. retrieval_evaluation_details.csv
2. retrieval_evaluation_summary.csv
3. answer_quality_evaluation.csv


In [15]:

# Verify Dataset and Saved Evaluation Files

import pandas as pd
import os

# Restore the original dataset
DATA_FILE = "Data.xml.xlsx"

if os.path.exists(DATA_FILE):
    df = pd.read_excel(DATA_FILE)
    df.columns = df.columns.str.strip()
    df = df.dropna(subset=["Problem", "Solution"]).reset_index(drop=True)
    df["Problem"] = df["Problem"].astype(str).str.strip()
    df["Solution"] = df["Solution"].astype(str).str.strip()
    df = df[
        (df["Problem"] != "") & (df["Solution"] != "")
    ].reset_index(drop=True)

    print("Original dataset restored.")
    print("Dataset records:", len(df))
    print("Dataset columns:", df.columns.tolist())
else:
    raise FileNotFoundError(f"{DATA_FILE} not found.")

# Verify all three evaluation files
evaluation_files = [
    "retrieval_evaluation_details.csv",
    "retrieval_evaluation_summary.csv",
    "answer_quality_evaluation.csv"
]

for filename in evaluation_files:
    print(f"\n--- {filename} ---")

    if os.path.exists(filename):
        saved_df = pd.read_csv(filename)
        print("Rows:", len(saved_df))
        display(saved_df.head())
    else:
        print("File not found in the current working directory.")


Original dataset restored.
Dataset records: 86
Dataset columns: ['Problem', 'Solution']

--- retrieval_evaluation_details.csv ---
Rows: 50


,Question,Method,Expected Problem,Accepted,Attempts,Matching Rank,Similarity Score,Recall@5,Reciprocal Rank
0,Python list index error,Fixed,Python list index error,True,1,1,0.868633,1,1.0
1,Python list index error,Semantic,Python list index error,True,1,1,0.811266,1,1.0
2,Python list index error,Recursive,Python list index error,True,1,1,0.811266,1,1.0
3,Python list index error,Documentation,Python list index error,True,1,1,0.811266,1,1.0
4,Python list index error,Agentic,Python list index error,True,1,1,0.811266,1,1.0



--- retrieval_evaluation_summary.csv ---
Rows: 5


,Method,Test_Questions,Successful_Retrievals,Recall_at_5,MRR
0,Fixed,10,10,100.0,1.0
1,Semantic,10,10,100.0,1.0
2,Recursive,10,10,100.0,1.0
3,Documentation,10,10,100.0,1.0
4,Agentic,10,10,100.0,1.0



--- answer_quality_evaluation.csv ---
Rows: 5


,Question,Generated Answer,Correctness,Faithfulness,Evaluation Reason
0,Python list index error,An **IndexError** occurs when you try to acces...,5,5,The generated answer accurately explains that ...
1,Python KeyError in dictionary,A **KeyError** in Python happens when you try ...,5,5,The answer accurately explains that a KeyError...
2,Python module not found,A ModuleNotFoundError usually means the packag...,5,5,The generated answer exactly matches the refer...
3,Python virtual environment activation,To activate a Python virtual environment in Po...,5,5,The answer accurately follows the reference so...
4,Python file path error,A **FileNotFoundError** in Python usually mean...,5,5,The generated answer accurately reflects the r...


## 13. Conclusion

In this project, we developed a Retrieval-Augmented Generation (RAG) system for question answering over structured Problem–Solution data. The system uses Sentence Transformers to generate embeddings, FAISS for similarity-based retrieval, and Groq to generate answers grounded in retrieved context.

We implemented and compared five retrieval configurations: Fixed, Semantic, Recursive, Documentation, and Agentic Retrieval. The Agentic Retrieval controller uses an iterative process to retrieve candidate records, assess similarity scores, refine the query when necessary, and reject results that do not meet a configured similarity threshold.

The evaluation used 10 predefined test questions and expected problem titles. All five methods achieved 100% Recall@5 and an MRR of 1.00 on this test set. These results indicate successful retrieval of the expected records for the selected questions; they do not establish that the methods perform equally well on unseen questions or that generated answers are always correct.

An additional out-of-scope test using a chocolate-cake question demonstrated that the Agentic controller rejected retrieval results below the configured similarity threshold of 0.60 after two attempts. However, similarity scores are not calibrated probabilities of relevance, and the threshold requires further validation.

**Limitations:** The dataset contains 86 Problem–Solution records, the evaluation uses only 10 predefined questions, and the retrieval metrics rely on expected-title matching. The Agentic controller is rule-based rather than a fully autonomous LLM agent. Generated-answer correctness and faithfulness were evaluated using an LLM-based evaluator on five predefined questions. These scores are preliminary and have not been independently validated by human reviewers.

**Future Work:** Future improvements include expanding the dataset and test questions, validating relevance thresholds, obtaining independent relevance judgments, evaluating generated-answer quality, and investigating LLM-driven query refinement or more advanced agentic retrieval strategies.